In [1]:
# Instaliramo najnoviju verziju Whisper-a direktno sa GitHub-a i ffmpeg za obradu audia
!pip install -q git+https://github.com/openai/whisper.git
!sudo apt-get -y install ffmpeg

print("✅ Instalacija završena!")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 110.9 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Read

In [2]:
from google.colab import files
uploaded_files = files.upload()
AUDIO_FILES = list(uploaded_files.keys())
print(f"Učitani fajlovi: {AUDIO_FILES}")

Saving serija.mp3 to serija.mp3
Saving monolog.mp3 to monolog.mp3
Saving dijalog.mp3 to dijalog.mp3
Učitani fajlovi: ['serija.mp3', 'monolog.mp3', 'dijalog.mp3']


In [3]:
import os
import whisper
import time
from datetime import datetime
import shutil  # Import za rad sa fajlovima i arhivama
from google.colab import files # Import za preuzimanje fajlova

# --- POČETAK KONFIGURACIJE ---

# Definišite koje modele želite koristiti. Preporučujem 'large-v3' kao najnoviji.
MODEL_SIZES = ["large", "large-v2", "large-v3"]

# Jezici nad kojim ću testirati (vaša originalna konfiguracija)
LANGUAGES = {
    "Bosanski": "bs",
    "Hrvatski": "hr",
    "Srpski": "sr"
}

# Naziv poddirektorija za spremanje rezultata
TIMESTAMP_DIR = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_BASE_DIRECTORY = "transkript_rezultati"
OUTPUT_DIRECTORY = os.path.join(OUTPUT_BASE_DIRECTORY, f"rezultati_{TIMESTAMP_DIR}")
# --- KRAJ KONFIGURACIJE ---

# VAŠE FUNKCIJE OSTAJU ISTE (nisam ih menjao)
def format_time(seconds):
    if seconds < 0: return "N/A"
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes}m {secs:.2f}s"

def save_result_files(output_dir, base_filename, lang_name, model_name,
                      transcription_time_seconds, formatted_time, transcription_text, error_occurred=False):
    try:
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        if error_occurred:
            error_filename = f"{base_filename}_{lang_name}_{model_name}_GREŠKA.txt"
            error_filepath = os.path.join(output_dir, error_filename)
            with open(error_filepath, 'w', encoding='utf-8') as f_error:
                f_error.write(f"Došlo je do greške tokom transkripcije.\nPoruka greške: {transcription_text}\n")
            print(f"      Informacije o grešci spremljene u: {error_filepath}")
        else:
            time_filename = f"{base_filename}_{lang_name}_{model_name}_vrijeme.txt"
            transcript_filename = f"{base_filename}_{lang_name}_{model_name}_transkript.txt"
            time_filepath = os.path.join(output_dir, time_filename)
            transcript_filepath = os.path.join(output_dir, transcript_filename)
            with open(time_filepath, 'w', encoding='utf-8') as f_time:
                f_time.write(f"Vrijeme transkripcije (s): {transcription_time_seconds:.2f}\nFormatirano vrijeme: {formatted_time}\n")
            print(f"      Vrijeme transkripcije spremljeno u: {time_filepath}")
            with open(transcript_filepath, 'w', encoding='utf-8') as f_transcript:
                f_transcript.write(transcription_text)
            print(f"      Transkript spremljen u: {transcript_filepath}")
    except Exception as e:
        print(f"      GREŠKA pri spremanju fajla za {base_filename}_{lang_name}_{model_name}: {e}")

def main():
    if not os.path.exists(OUTPUT_DIRECTORY):
        os.makedirs(OUTPUT_DIRECTORY)
    print(f"Rezultati će biti spremljeni u: {OUTPUT_DIRECTORY}")
    print("\n--- Pokretanje procesa transkripcije ---")

    # Lista fajlova se sada uzima iz promenljive AUDIO_FILES definisane u prethodnoj ćeliji
    for audio_file_full_path in AUDIO_FILES:
        if not os.path.exists(audio_file_full_path):
            print(f"GREŠKA: Fajl '{audio_file_full_path}' nije pronađen. Preskačem.")
            continue
        audio_filename_base = os.path.splitext(os.path.basename(audio_file_full_path))[0]
        print(f"\n>>> Obrada fajla: {audio_file_full_path} <<<")

        for model_name in MODEL_SIZES:
            print(f"  Učitavanje modela: {model_name}...")
            model = None
            try:
                # Colab GPU će ovo učitati bez problema
                model = whisper.load_model(model_name)
            except Exception as e:
                print(f"    GREŠKA pri učitavanju modela {model_name}: {e}")
                save_result_files(OUTPUT_DIRECTORY, audio_filename_base, "N/A", model_name, -1, "N/A", str(e), error_occurred=True)
                print(f"    Preskačem sve jezike za model '{model_name}' i fajl '{audio_filename_base}'.")
                continue

            for lang_name, lang_code in LANGUAGES.items():
                print(f"    -> Transkripcija za jezik: {lang_name} ({lang_code})")
                start_time = time.time()
                try:
                    result = model.transcribe(audio_file_full_path, language=lang_code, fp16=True) # fp16=True ubrzava na GPU
                    transcription_text = result["text"]
                    end_time = time.time()
                    transcription_time = end_time - start_time
                    formatted_time = format_time(transcription_time)
                    print(f"      Završeno za {formatted_time}")
                    save_result_files(OUTPUT_DIRECTORY, audio_filename_base, lang_name, model_name, transcription_time, formatted_time, transcription_text)
                except Exception as e:
                    print(f"      GREŠKA tokom transkripcije: {e}")
                    save_result_files(OUTPUT_DIRECTORY, audio_filename_base, lang_name, model_name, -1, "N/A", str(e), error_occurred=True)

    print("\n--- Transkripcija završena za sve fajlove ---")

    # *** KLJUČNI DEO: Pakovanje rezultata u ZIP i preuzimanje ***
    if os.path.exists(OUTPUT_DIRECTORY):
        print("\nPakovanje rezultata u ZIP arhivu...")
        zip_filename = f"rezultati_{TIMESTAMP_DIR}.zip"
        shutil.make_archive(f"rezultati_{TIMESTAMP_DIR}", 'zip', OUTPUT_BASE_DIRECTORY)
        print(f"Arhiva '{zip_filename}' je kreirana. Pokrećem preuzimanje...")
        files.download(zip_filename)
    else:
        print("Direktorijum sa rezultatima nije kreiran, nema šta za preuzimanje.")


# Pokretanje glavne funkcije
if 'AUDIO_FILES' in locals() and AUDIO_FILES:
    main()
else:
    print("Niste uploadovali nijedan fajl. Pokrenite prethodnu ćeliju i uploadujte audio fajlove.")

Rezultati će biti spremljeni u: transkript_rezultati/rezultati_20250702_121032

--- Pokretanje procesa transkripcije ---

>>> Obrada fajla: serija.mp3 <<<
  Učitavanje modela: large...


100%|█████████████████████████████████████| 2.88G/2.88G [01:44<00:00, 29.5MiB/s]


    -> Transkripcija za jezik: Bosanski (bs)
      Završeno za 2m 6.99s
      Vrijeme transkripcije spremljeno u: transkript_rezultati/rezultati_20250702_121032/serija_Bosanski_large_vrijeme.txt
      Transkript spremljen u: transkript_rezultati/rezultati_20250702_121032/serija_Bosanski_large_transkript.txt
    -> Transkripcija za jezik: Hrvatski (hr)
      Završeno za 2m 15.94s
      Vrijeme transkripcije spremljeno u: transkript_rezultati/rezultati_20250702_121032/serija_Hrvatski_large_vrijeme.txt
      Transkript spremljen u: transkript_rezultati/rezultati_20250702_121032/serija_Hrvatski_large_transkript.txt
    -> Transkripcija za jezik: Srpski (sr)
      Završeno za 2m 27.02s
      Vrijeme transkripcije spremljeno u: transkript_rezultati/rezultati_20250702_121032/serija_Srpski_large_vrijeme.txt
      Transkript spremljen u: transkript_rezultati/rezultati_20250702_121032/serija_Srpski_large_transkript.txt
  Učitavanje modela: large-v2...


100%|█████████████████████████████████████| 2.87G/2.87G [00:49<00:00, 62.8MiB/s]


    -> Transkripcija za jezik: Bosanski (bs)
      Završeno za 1m 6.07s
      Vrijeme transkripcije spremljeno u: transkript_rezultati/rezultati_20250702_121032/serija_Bosanski_large-v2_vrijeme.txt
      Transkript spremljen u: transkript_rezultati/rezultati_20250702_121032/serija_Bosanski_large-v2_transkript.txt
    -> Transkripcija za jezik: Hrvatski (hr)
      Završeno za 1m 0.78s
      Vrijeme transkripcije spremljeno u: transkript_rezultati/rezultati_20250702_121032/serija_Hrvatski_large-v2_vrijeme.txt
      Transkript spremljen u: transkript_rezultati/rezultati_20250702_121032/serija_Hrvatski_large-v2_transkript.txt
    -> Transkripcija za jezik: Srpski (sr)
      Završeno za 1m 15.85s
      Vrijeme transkripcije spremljeno u: transkript_rezultati/rezultati_20250702_121032/serija_Srpski_large-v2_vrijeme.txt
      Transkript spremljen u: transkript_rezultati/rezultati_20250702_121032/serija_Srpski_large-v2_transkript.txt
  Učitavanje modela: large-v3...
    -> Transkripcija za jez

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>